### Setup

In [1]:
import time
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

In [2]:
model_name = "distilgpt2"  # small + fast

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token 

model = AutoModelForCausalLM.from_pretrained(model_name)
model.eval()
print(model_name)

distilgpt2


### Utility: Simple Cost Tracker

In [3]:
def run_inference(prompts, max_new_tokens=30):
    start = time.time()

    inputs = tokenizer(prompts, return_tensors="pt", padding=True)
    input_tokens = inputs["input_ids"].numel()

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens
        )

    output_tokens = outputs.numel()
    latency = time.time() - start

    return {
        "input_tokens": input_tokens,
        "output_tokens": output_tokens,
        "latency": latency,
        "responses": tokenizer.batch_decode(outputs, skip_special_tokens=True)
    }

### Baseline: No Optimization (Sequential Requests)

In [4]:
prompts = [
    "Explain machine learning in simple terms.",
    "Explain machine learning in simple terms.",
    "Explain machine learning in simple terms."
]

results = []
for p in prompts:
    results.append(run_inference(p))

baseline_cost = {
    "input_tokens": sum(r["input_tokens"] for r in results),
    "output_tokens": sum(r["output_tokens"] for r in results),
    "latency": sum(r["latency"] for r in results)
}

baseline_cost

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


{'input_tokens': 24, 'output_tokens': 114, 'latency': 3.9672842025756836}

### Optimization 1: Prompt Compression

In [5]:
# Original prompt: Explain machine learning in simple terms.
# Compressed prompt: Explain ML simply.

compressed_prompt = "Explain ML simply."
run_inference(compressed_prompt)

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


{'input_tokens': 5,
 'output_tokens': 35,
 'latency': 1.5687570571899414,
 'responses': ['Explain ML simply.\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n']}

### Optimization 2: Response Caching

In [6]:
cache = {}

def cached_inference(prompt):
    if prompt in cache:
        return cache[prompt], True  # cache hit
    result = run_inference(prompt)
    cache[prompt] = result
    return result, False

In [7]:
hits = 0
for p in prompts:
    _, hit = cached_inference(p)
    hits += int(hit)

hits

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


2

### Optimization 3: Request Batching

In [8]:
batched_result = run_inference(prompts)
batched_result

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


{'input_tokens': 24,
 'output_tokens': 114,
 'latency': 2.3608241081237793,
 'responses': ['Explain machine learning in simple terms.\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n',
  'Explain machine learning in simple terms.\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n',
  'Explain machine learning in simple terms.\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n']}

In [9]:
# Compare latency:
print("Baseline latency:", baseline_cost["latency"])
print("Batched latency:", batched_result["latency"])

Baseline latency: 3.9672842025756836
Batched latency: 2.3608241081237793


### Optimization 4: Quantization (Simple Demo)

In [10]:
model_fp16 = AutoModelForCausalLM.from_pretrained(
    model_name,
    dtype=torch.float16
)

model_fp16.eval()

GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-5): 6 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=2304, nx=768)
          (c_proj): Conv1D(nf=768, nx=768)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=3072, nx=768)
          (c_proj): Conv1D(nf=768, nx=3072)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=768, out_features=50257, bias=False)
)

In [11]:
def run_fp16(prompt):
    inputs = tokenizer(prompt, return_tensors="pt")
    with torch.no_grad():
        outputs = model_fp16.generate(**inputs, max_new_tokens=30)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

### Before vs After: Cost Comparison

In [12]:
optimized_cost = {
    "input_tokens": batched_result["input_tokens"],
    "output_tokens": batched_result["output_tokens"],
    "latency": batched_result["latency"]
}

print("Baseline:", baseline_cost)
print("Optimized:", optimized_cost)

Baseline: {'input_tokens': 24, 'output_tokens': 114, 'latency': 3.9672842025756836}
Optimized: {'input_tokens': 24, 'output_tokens': 114, 'latency': 2.3608241081237793}
